In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable

silver_table = "weather_stream.silver.weather_data"
gold_avg_table = "weather_stream.gold.metrics_avg"
gold_min_table = "weather_stream.gold.metrics_min"
gold_max_table = "weather_stream.gold.metrics_max"
gold_stddev_table = "weather_stream.gold.metrics_stddev"
checkpoint_path = "/Volumes/weather_stream/infra/streaming_checkpoints_v2"

spark.sql("CREATE SCHEMA IF NOT EXISTS weather_stream.gold")

In [0]:
dbutils.fs.rm("/Volumes/weather_stream/infra/streaming_checkpoints", recurse=True)
dbutils.fs.rm("/Volumes/weather_stream/infra/streaming_checkpoints_v2", recurse=True)
print("Чекпоинты очищены.")

In [0]:
numeric_columns = [
    "AirPres", "Ambient_temperature", "Brake_active", "Error_code",
    "Grid_connected", "Humidity", "Operation_mode", "Power_output",
    "Rotor_rpm", "Wind_speed"
]

# 4. Чтение потока из Silver (Change Data Feed, только новые вставки)
silver_stream = (spark.readStream
    .format("delta")
    .option("readChangeFeed", "true")
    .option("startingVersion", "latest")
    .table(silver_table)
    .where("_change_type = 'insert'")
    .drop("_change_type", "_commit_version", "_commit_timestamp")
)

# 5. Watermark (1 минута) для обработки поздних данных
silver_stream = silver_stream.withWatermark("event_time", "1 minute")

# 6. Построение агрегаций
agg_exprs = []
for c in numeric_columns:
    agg_exprs.append(avg(c).alias(f"{c}_avg"))
    agg_exprs.append(min(c).alias(f"{c}_min"))
    agg_exprs.append(max(c).alias(f"{c}_max"))
    agg_exprs.append(stddev(c).alias(f"{c}_stddev"))

# 7. Оконная агрегация по 1 минуте
aggregated_df = silver_stream \
    .groupBy(window("event_time", "1 minute", "1 minute").alias("window"), "device_id") \
    .agg(*agg_exprs)

# 8. Извлечение начала и конца окна
aggregated_df = aggregated_df \
    .withColumn("window_start", col("window.start")) \
    .withColumn("window_end", col("window.end")) \
    .drop("window") \
    .withColumn("gold_inserted_at", current_timestamp())

# 9. Функция разделения по суффиксам (avg, min, max, stddev)
def split_by_suffix(df, suffix):
    cols = ["device_id", "window_start", "window_end", "gold_inserted_at"]
    for c in df.columns:
        if c.endswith(suffix):
            new_name = c.replace(suffix, "")
            cols.append(col(c).alias(new_name))
    return df.select(*cols)

# 10. Применяем разделение
avg_df = split_by_suffix(aggregated_df, "_avg")
min_df = split_by_suffix(aggregated_df, "_min")
max_df = split_by_suffix(aggregated_df, "_max")
stddev_df = split_by_suffix(aggregated_df, "_stddev")

# 12. Запись потоков в Gold таблицы (append)
query_avg = avg_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", f"{checkpoint_path}/avg") \
    .trigger(availableNow=True) \
    .table(gold_avg_table)

query_min = min_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", f"{checkpoint_path}/min") \
    .trigger(availableNow=True) \
    .table(gold_min_table)

query_max = max_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", f"{checkpoint_path}/max") \
    .trigger(availableNow=True) \
    .table(gold_max_table)

query_stddev = stddev_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", f"{checkpoint_path}/stddev") \
    .trigger(availableNow=True) \
    .table(gold_stddev_table)

# 13. Ожидание завершения всех потоков
spark.streams.awaitAnyTermination()
print("✅ Gold стриминг завершил обработку доступных новых данных.")